# `Dynamic AI Knowledge Assistant`

In [31]:
# ============================================================
# STEP 1 - PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNK_METADATA_DIR = ( PROCESSED_DIR / "chunk_metadata")
EMBEDDINGS_DIR = ( PROCESSED_DIR / "embeddings")
VECTOR_DB_DIR = (PROCESSED_DIR / "vector_db")
CHROMA_DB_DIR = (VECTOR_DB_DIR / "chroma")
CHUNK_METADATA_PATH = (CHUNK_METADATA_DIR / "chunk_metadata.csv")
EMBEDDINGS_PATH = (EMBEDDINGS_DIR /"all_MiniLM_L6_v2_embeddings.npy")

print("=" * 70)
print("PHASE 7 – PROJECT CONFIGURATION")
print("=" * 70)

print("Project root:", PROJECT_ROOT)
print("Chunk metadata:", CHUNK_METADATA_PATH)
print("Embeddings:", EMBEDDINGS_PATH)
print("ChromaDB:", CHROMA_DB_DIR)

# ============================================================
# STEP 2 -  VERIFY PHASE 6 ARTIFACTS
# ============================================================

required_paths = {
    "Project root": PROJECT_ROOT,
    "Chunk metadata": CHUNK_METADATA_PATH,
    "Embeddings": EMBEDDINGS_PATH,
    "ChromaDB directory": CHROMA_DB_DIR
}

print("=" * 70)
print("PHASE 6 ARTIFACT VERIFICATION")
print("=" * 70)

for name, path in required_paths.items():

    print(f"{name:<25}: {path.exists()}")

    if not path.exists():
        print("  MISSING:", path)

PHASE 7 – PROJECT CONFIGURATION
Project root: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant
Chunk metadata: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunk_metadata\chunk_metadata.csv
Embeddings: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_MiniLM_L6_v2_embeddings.npy
ChromaDB: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\chroma
PHASE 6 ARTIFACT VERIFICATION
Project root             : True
Chunk metadata           : True
Embeddings               : True
ChromaDB directory       : True


In [32]:
# ============================================================
# STEP 3 -  LOAD CHUNK METADATA
# ============================================================

chunk_metadata_df = pd.read_csv(CHUNK_METADATA_PATH)

print("=" * 70)
print("CHUNK METADATA")
print("=" * 70)

print("Shape:", chunk_metadata_df.shape)
print("Columns:", chunk_metadata_df.columns.tolist())
print("\nFirst 5 records:")
display(chunk_metadata_df.head())

CHUNK METADATA
Shape: (4693, 9)
Columns: ['chunk_id', 'article_id', 'title', 'category', 'source_url', 'original_file_name', 'chunk_index', 'chunk_text', 'chunk_length']

First 5 records:


,chunk_id,article_id,title,category,source_url,original_file_name,chunk_index,chunk_text,chunk_length
0,ART_84789a287028_chunk_0000,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,0,A 1.58-bit large language model (also known as...,809
1,ART_84789a287028_chunk_0001,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,1,"The name ""1.58-bit"" comes from the fact that a...",319
2,ART_84789a287028_chunk_0002,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,2,"== BitNet == In 2024, Ma et al., researchers a...",600
3,ART_84789a287028_chunk_0003,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,3,== Post-training quantization == BitNet derive...,438
4,ART_84789a287028_chunk_0004,ART_84789a287028,1.58-bit large language model,Large Language Model,https://en.wikipedia.org/wiki/1.58-bit_large_l...,1.58-bit_large_language_model.txt,4,== Critique == Some researchers point out that...,245


In [33]:
# ============================================================
# STEP 4 -  VALIDATE CHUNK METADATA
# ============================================================

EXPECTED_COLUMNS = [
    "chunk_id",
    "article_id",
    "title",
    "category",
    "source_url",
    "original_file_name",
    "chunk_index",
    "chunk_text",
    "chunk_length"
]

assert chunk_metadata_df.shape[0] == 4693

assert (
    chunk_metadata_df.columns.tolist()
    == EXPECTED_COLUMNS
)

assert chunk_metadata_df["chunk_id"].is_unique

assert chunk_metadata_df["chunk_text"].notna().all()

assert (
    chunk_metadata_df["chunk_text"]
    .str.len()
    .gt(0)
    .all()
)

print("Chunk count          : PASSED")
print("Schema               : PASSED")
print("Unique chunk IDs     : PASSED")
print("Chunk text           : PASSED")

Chunk count          : PASSED
Schema               : PASSED
Unique chunk IDs     : PASSED
Chunk text           : PASSED


In [34]:
# ============================================================
# STEP 5 - LOAD EMBEDDINGS
# ============================================================

embeddings = np.load(EMBEDDINGS_PATH)

print("=" * 70)
print("EMBEDDINGS")
print("=" * 70)

print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)
assert embeddings.shape == (4693, 384)

assert embeddings.shape[0] == len(
    chunk_metadata_df
)

print("\nEmbedding count       : PASSED")
print("Embedding dimensions  : PASSED")
print("Metadata alignment    : PASSED")

EMBEDDINGS
Shape: (4693, 384)
Dtype: float32

Embedding count       : PASSED
Embedding dimensions  : PASSED
Metadata alignment    : PASSED


In [35]:
# ============================================================
# STEP 6 -  LOAD CHROMADB
# ============================================================

import chromadb

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))

print("=" * 70)
print("CHROMADB")
print("=" * 70)
print("Available collections:")
collections = chroma_client.list_collections()
for collection_info in collections:
    print("-", collection_info.name)

collection = chroma_client.get_collection(
    name="wiki_chunks_minilm"
)
print("\nCollection count:")
print(collection.count())

CHROMADB
Available collections:
- wiki_chunks_minilm

Collection count:
4693


In [36]:
# ============================================================
#  STEP 7 -  LOAD EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embedding_dimension = (
    model.get_embedding_dimension()
)

print("=" * 70)
print("EMBEDDING MODEL")
print("=" * 70)

print("Model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_dimension)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7715.35it/s]


EMBEDDING MODEL
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [37]:
# ============================================================
# STEP 8 - QUERY EMBEDDING FUNCTION
# ============================================================

def generate_query_embedding(query):
    """
    Convert a user query into a normalized embedding.
    """

    if not isinstance(query, str):
        raise TypeError("Query must be a string.")

    query = query.strip()

    if not query:
        raise ValueError("Query cannot be empty.")

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return query_embedding.astype(np.float32)


test_query = "What is artificial intelligence?"
query_embedding = generate_query_embedding(test_query)
print("Query:", test_query)
print("Query embedding shape:", query_embedding.shape)
print("Dtype:", query_embedding.dtype)

assert query_embedding.shape == (1, 384)
assert query_embedding.dtype == np.float32
assert np.isfinite(query_embedding).all()

print("Query embedding shape : PASSED")
print("Embedding dtype       : PASSED")
print("Finite values         : PASSED")

Query: What is artificial intelligence?
Query embedding shape: (1, 384)
Dtype: float32
Query embedding shape : PASSED
Embedding dtype       : PASSED
Finite values         : PASSED


In [38]:
# ============================================================
#  STEP 9 - BASIC SIMILARITY SEARCH
# ============================================================

def basic_similarity_search(query, top_k=5):
    """
    Retrieve the top-K semantically similar chunks
    from the existing ChromaDB collection.
    """

    # --------------------------------------------------------
    # Validate query
    # --------------------------------------------------------

    if not isinstance(query, str):
        raise TypeError("Query must be a string.")

    query = query.strip()

    if not query:
        raise ValueError("Query cannot be empty.")

    # --------------------------------------------------------
    # Validate top_k
    # --------------------------------------------------------

    if not isinstance(top_k, int):
        raise TypeError("top_k must be an integer.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero.")

    collection_count = collection.count()

    if top_k > collection_count:
        raise ValueError(
            f"top_k cannot exceed collection size "
            f"({collection_count})."
        )

    # --------------------------------------------------------
    # Generate query embedding
    # --------------------------------------------------------

    query_embedding = generate_query_embedding(query)

    # --------------------------------------------------------
    # Query ChromaDB
    # --------------------------------------------------------

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    # --------------------------------------------------------
    # Extract results
    # --------------------------------------------------------

    ids = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    # --------------------------------------------------------
    # Convert cosine distance to cosine similarity
    #
    # cosine similarity = 1 - cosine distance
    # --------------------------------------------------------

    similarities = [
        1 - distance
        for distance in distances
    ]

    # --------------------------------------------------------
    # Build retrieval records
    # --------------------------------------------------------

    records = []

    for rank in range(len(ids)):

        metadata = metadatas[rank]

        records.append({
            "rank": rank + 1,
            "chunk_id": ids[rank],
            "article_id": metadata["article_id"],
            "title": metadata["title"],
            "category": metadata["category"],
            "source_url": metadata["source_url"],
            "original_file_name": metadata[
                "original_file_name"
            ],
            "chunk_index": metadata[
                "chunk_index"
            ],
            "chunk_text": documents[rank],
            "similarity_score": similarities[rank]
        })

    return pd.DataFrame(records)

# ============================================================
# TEST BASIC SIMILARITY SEARCH
# ============================================================

query = "What is artificial intelligence?"

results_df = basic_similarity_search(
    query=query,
    top_k=5
)

print("=" * 70)
print("BASIC SIMILARITY SEARCH")
print("=" * 70)

print("Query:")
print(query)

print("\nNumber of retrieved chunks:")
print(len(results_df))

display(
    results_df[
        [
            "rank",
            "chunk_id",
            "article_id",
            "title",
            "chunk_index",
            "similarity_score"
        ]
    ]
)

BASIC SIMILARITY SEARCH
Query:
What is artificial intelligence?

Number of retrieved chunks:
5


,rank,chunk_id,article_id,title,chunk_index,similarity_score
0,1,ART_ede7794b399f_chunk_0000,ART_ede7794b399f,Artificial intelligence,0,0.780551
1,2,ART_7b50a3a1d378_chunk_0018,ART_7b50a3a1d378,Glossary of artificial intelligence,18,0.777261
2,3,ART_7b50a3a1d378_chunk_0000,ART_7b50a3a1d378,Glossary of artificial intelligence,0,0.734774
3,4,ART_5b75b0467a43_chunk_0046,ART_5b75b0467a43,Computer science,46,0.720271
4,5,ART_ede7794b399f_chunk_0167,ART_ede7794b399f,Artificial intelligence,167,0.714266


In [39]:
# ============================================================
# STEP 9 – RETRIEVAL VALIDATION
# ============================================================

assert len(results_df) == 5

required_columns = {
    "rank",
    "chunk_id",
    "article_id",
    "title",
    "category",
    "source_url",
    "original_file_name",
    "chunk_index",
    "chunk_text",
    "similarity_score"
}

assert required_columns.issubset(
    results_df.columns
)

assert results_df["chunk_id"].notna().all()

assert results_df["chunk_id"].is_unique

assert results_df["chunk_text"].notna().all()

assert results_df["chunk_text"].str.len().gt(0).all()

assert results_df["source_url"].notna().all()

assert results_df["similarity_score"].between(
    -1, 1
).all()

assert results_df[
    "similarity_score"
].is_monotonic_decreasing


print("=" * 70)
print("STEP 9 – BASIC SIMILARITY SEARCH VALIDATION")
print("=" * 70)

print("Top-K retrieval          : PASSED")
print("Retrieval schema         : PASSED")
print("Unique chunk IDs         : PASSED")
print("Chunk text               : PASSED")
print("Source URLs              : PASSED")
print("Similarity scores       : PASSED")
print("Similarity ranking      : PASSED")

STEP 9 – BASIC SIMILARITY SEARCH VALIDATION
Top-K retrieval          : PASSED
Retrieval schema         : PASSED
Unique chunk IDs         : PASSED
Chunk text               : PASSED
Source URLs              : PASSED
Similarity scores       : PASSED
Similarity ranking      : PASSED


In [40]:
# ============================================================
# STEP 10 -  INSPECT RETRIEVED CONTEXT
# ============================================================

print("=" * 80)
print("RETRIEVED CONTEXT INSPECTION")
print("=" * 80)

print("\nQuery:")
print(query)

for _, row in results_df.iterrows():

    print("\n" + "=" * 80)
    print(f"RANK {row['rank']}")
    print("=" * 80)

    print("Title:", row["title"])
    print("Chunk ID:", row["chunk_id"])
    print("Article ID:", row["article_id"])
    print("Category:", row["category"])
    print("Chunk Index:", row["chunk_index"])
    print(
        "Similarity Score:",
        round(row["similarity_score"], 6)
    )
    print("Source URL:", row["source_url"])

    print("\nChunk Text:")
    print(row["chunk_text"])

# ============================================================
# COMPACT RETRIEVAL VIEW
# ============================================================

retrieval_view = results_df[
    [
        "rank",
        "title",
        "category",
        "chunk_index",
        "similarity_score",
        "source_url"
    ]
].copy()

retrieval_view["similarity_score"] = (
    retrieval_view["similarity_score"]
    .round(6)
)

display(retrieval_view)


# ============================================================
# SIMILARITY ORDER VALIDATION
# ============================================================

scores = results_df["similarity_score"].to_numpy()

assert np.all(
    scores[:-1] >= scores[1:]
)

print("Similarity ordering: PASSED")

# ============================================================
# RETRIEVAL STATISTICS
# ============================================================

print("=" * 70)
print("RETRIEVAL STATISTICS")
print("=" * 70)

print("Query:", query)
print("Top-K:", len(results_df))

print(
    "Highest similarity:",
    round(
        results_df["similarity_score"].max(),
        6
    )
)

print(
    "Lowest similarity:",
    round(
        results_df["similarity_score"].min(),
        6
    )
)

print(
    "Average similarity:",
    round(
        results_df["similarity_score"].mean(),
        6
    )
)

print(
    "Unique articles retrieved:",
    results_df["article_id"].nunique()
)

RETRIEVED CONTEXT INSPECTION

Query:
What is artificial intelligence?

RANK 1
Title: Artificial intelligence
Chunk ID: ART_ede7794b399f_chunk_0000
Article ID: ART_ede7794b399f
Category: Artificial Intelligence
Chunk Index: 0
Similarity Score: 0.780551
Source URL: https://en.wikipedia.org/wiki/Artificial_intelligence

Chunk Text:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals. High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go

,rank,title,category,chunk_index,similarity_score,source_url
0,1,Artificial intelligence,Artificial Intelligence,0,0.780551,https://en.wikipedia.org/wiki/Artificial_intel...
1,2,Glossary of artificial intelligence,Artificial Intelligence,18,0.777261,https://en.wikipedia.org/wiki/Glossary_of_arti...
2,3,Glossary of artificial intelligence,Artificial Intelligence,0,0.734774,https://en.wikipedia.org/wiki/Glossary_of_arti...
3,4,Computer science,Data Science,46,0.720271,https://en.wikipedia.org/wiki/Computer_science
4,5,Artificial intelligence,Artificial Intelligence,167,0.714266,https://en.wikipedia.org/wiki/Artificial_intel...


Similarity ordering: PASSED
RETRIEVAL STATISTICS
Query: What is artificial intelligence?
Top-K: 5
Highest similarity: 0.780551
Lowest similarity: 0.714266
Average similarity: 0.745425
Unique articles retrieved: 3


In [41]:
# ============================================================
# STEP 10-  VALIDATION
# ============================================================

assert len(results_df) == 5

assert results_df["chunk_text"].notna().all()
assert results_df["chunk_text"].str.len().gt(0).all()

assert results_df["source_url"].notna().all()
assert results_df["article_id"].notna().all()
assert results_df["title"].notna().all()

assert results_df["chunk_id"].is_unique

scores = results_df["similarity_score"].to_numpy()

assert np.all(
    scores[:-1] >= scores[1:]
)

print("=" * 70)
print("STEP 10 – RETRIEVED CONTEXT VALIDATION")
print("=" * 70)

print("Retrieved context        : PASSED")
print("Chunk traceability       : PASSED")
print("Source preservation      : PASSED")
print("Non-empty chunk text     : PASSED")
print("Unique retrieved chunks  : PASSED")
print("Similarity ordering      : PASSED")

# ============================================================
# RETRIEVAL STATISTICS
# ============================================================

print("=" * 70)
print("RETRIEVAL STATISTICS")
print("=" * 70)

print("Query:", query)
print("Top-K:", len(results_df))

print(
    "Highest similarity:",
    round(results_df["similarity_score"].max(), 6)
)

print(
    "Lowest similarity:",
    round(results_df["similarity_score"].min(), 6)
)

print(
    "Average similarity:",
    round(results_df["similarity_score"].mean(), 6)
)

print(
    "Unique articles retrieved:",
    results_df["article_id"].nunique()
)

STEP 10 – RETRIEVED CONTEXT VALIDATION
Retrieved context        : PASSED
Chunk traceability       : PASSED
Source preservation      : PASSED
Non-empty chunk text     : PASSED
Unique retrieved chunks  : PASSED
Similarity ordering      : PASSED
RETRIEVAL STATISTICS
Query: What is artificial intelligence?
Top-K: 5
Highest similarity: 0.780551
Lowest similarity: 0.714266
Average similarity: 0.745425
Unique articles retrieved: 3


In [42]:
# ============================================================
#  STEP 11 -  RETRIEVAL LATENCY BENCHMARK
# ============================================================

import time
import numpy as np
import pandas as pd

# ============================================================
# BENCHMARK CONFIGURATION
# ============================================================

benchmark_queries = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?",
    "What is a large language model?"
]

TOP_K = 5
WARMUP_RUNS = 1
BENCHMARK_RUNS = 5

print("Benchmark queries:", len(benchmark_queries))
print("Top-K:", TOP_K)
print("Warm-up runs:", WARMUP_RUNS)
print("Benchmark runs:", BENCHMARK_RUNS)

# ============================================================
# RUN RETRIEVAL BENCHMARK
# ============================================================

benchmark_results = []

for query_text in benchmark_queries:

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    for _ in range(WARMUP_RUNS):
        basic_similarity_search(
            query=query_text,
            top_k=TOP_K
        )

    # --------------------------------------------------------
    # Timed runs
    # --------------------------------------------------------

    run_times = []

    for _ in range(BENCHMARK_RUNS):

        start_time = time.perf_counter()

        result = basic_similarity_search(
            query=query_text,
            top_k=TOP_K
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time

        run_times.append(elapsed)

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    benchmark_results.append({
        "query": query_text,
        "runs": BENCHMARK_RUNS,
        "top_k": TOP_K,
        "min_latency_seconds": min(run_times),
        "max_latency_seconds": max(run_times),
        "mean_latency_seconds": np.mean(run_times),
        "median_latency_seconds": np.median(run_times)
    })

latency_df = pd.DataFrame(benchmark_results)

print("=" * 80)
print("RETRIEVAL LATENCY BENCHMARK")
print("=" * 80)

display(latency_df)

Benchmark queries: 5
Top-K: 5
Warm-up runs: 1
Benchmark runs: 5
RETRIEVAL LATENCY BENCHMARK


,query,runs,top_k,min_latency_seconds,max_latency_seconds,mean_latency_seconds,median_latency_seconds
0,What is artificial intelligence?,5,5,0.017974,0.021556,0.019998,0.019607
1,What is machine learning?,5,5,0.017771,0.044203,0.029741,0.031026
2,What is deep learning?,5,5,0.021122,0.041591,0.027222,0.023408
3,What is natural language processing?,5,5,0.023687,0.027165,0.025829,0.025741
4,What is a large language model?,5,5,0.018494,0.019834,0.019053,0.018652


In [43]:
# ============================================================
# OVERALL LATENCY STATISTICS
# ============================================================

all_mean_latencies = (
    latency_df["mean_latency_seconds"]
)

overall_mean = all_mean_latencies.mean()
overall_median = (
    latency_df["median_latency_seconds"].median()
)
overall_min = (
    latency_df["min_latency_seconds"].min()
)
overall_max = (
    latency_df["max_latency_seconds"].max()
)

print("=" * 80)
print("OVERALL RETRIEVAL LATENCY")
print("=" * 80)

print(
    "Mean latency   :",
    round(overall_mean, 4),
    "seconds"
)

print(
    "Median latency :",
    round(overall_median, 4),
    "seconds"
)

print(
    "Minimum latency:",
    round(overall_min, 4),
    "seconds"
)

print(
    "Maximum latency:",
    round(overall_max, 4),
    "seconds"
)

OVERALL RETRIEVAL LATENCY
Mean latency   : 0.0244 seconds
Median latency : 0.0234 seconds
Minimum latency: 0.0178 seconds
Maximum latency: 0.0442 seconds


In [44]:
# ============================================================
# STEP 11 VALIDATION
# ============================================================

assert not latency_df.empty

assert len(latency_df) == len(
    benchmark_queries
)

assert (
    latency_df["mean_latency_seconds"]
    .notna()
    .all()
)

assert (
    latency_df["mean_latency_seconds"]
    >= 0
).all()

assert (
    latency_df["median_latency_seconds"]
    >= 0
).all()

assert overall_mean >= 0
assert overall_median >= 0

print("=" * 70)
print("STEP 11 – RETRIEVAL LATENCY VALIDATION")
print("=" * 70)

print("Benchmark queries       : PASSED")
print("Latency measurements    : PASSED")
print("Non-negative latency    : PASSED")
print("Overall statistics      : PASSED")

STEP 11 – RETRIEVAL LATENCY VALIDATION
Benchmark queries       : PASSED
Latency measurements    : PASSED
Non-negative latency    : PASSED
Overall statistics      : PASSED


In [45]:
# ============================================================
# STEP 12 -  MULTI-QUERY RETRIEVAL TESTING
# ============================================================

test_queries = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?",
    "What is a large language model?",
    "What is a vector database?",
    "What is retrieval augmented generation?",
    "What is computer vision?"
]

TOP_K = 5

multi_query_results = []

for query_text in test_queries:

    start_time = time.perf_counter()

    results = basic_similarity_search(
        query=query_text,
        top_k=TOP_K
    )

    end_time = time.perf_counter()

    latency = end_time - start_time

    for _, row in results.iterrows():

        multi_query_results.append({
            "query": query_text,
            "rank": row["rank"],
            "chunk_id": row["chunk_id"],
            "article_id": row["article_id"],
            "title": row["title"],
            "chunk_index": row["chunk_index"],
            "similarity_score": row["similarity_score"],
            "source_url": row["source_url"],
            "latency_seconds": latency
        })

multi_query_df = pd.DataFrame(multi_query_results)

print("=" * 80)
print("MULTI-QUERY RETRIEVAL TESTING")
print("=" * 80)

print("Queries tested:", len(test_queries))
print("Top-K:", TOP_K)
print("Total retrieved results:", len(multi_query_df))

display(multi_query_df)

MULTI-QUERY RETRIEVAL TESTING
Queries tested: 8
Top-K: 5
Total retrieved results: 40


,query,rank,chunk_id,article_id,title,chunk_index,similarity_score,source_url,latency_seconds
0,What is artificial intelligence?,1,ART_ede7794b399f_chunk_0000,ART_ede7794b399f,Artificial intelligence,0,0.780551,https://en.wikipedia.org/wiki/Artificial_intel...,0.021215
1,What is artificial intelligence?,2,ART_7b50a3a1d378_chunk_0018,ART_7b50a3a1d378,Glossary of artificial intelligence,18,0.777261,https://en.wikipedia.org/wiki/Glossary_of_arti...,0.021215
2,What is artificial intelligence?,3,ART_7b50a3a1d378_chunk_0000,ART_7b50a3a1d378,Glossary of artificial intelligence,0,0.734774,https://en.wikipedia.org/wiki/Glossary_of_arti...,0.021215
3,What is artificial intelligence?,4,ART_5b75b0467a43_chunk_0046,ART_5b75b0467a43,Computer science,46,0.720271,https://en.wikipedia.org/wiki/Computer_science,0.021215
4,What is artificial intelligence?,5,ART_ede7794b399f_chunk_0167,ART_ede7794b399f,Artificial intelligence,167,0.714266,https://en.wikipedia.org/wiki/Artificial_intel...,0.021215
5,What is machine learning?,1,ART_ede7794b399f_chunk_0011,ART_ede7794b399f,Artificial intelligence,11,0.693090,https://en.wikipedia.org/wiki/Artificial_intel...,0.017185
6,What is machine learning?,2,ART_101732b69dfa_chunk_0049,ART_101732b69dfa,Neural network (machine learning),49,0.672806,https://en.wikipedia.org/wiki/Neural_network_(...,0.017185
7,What is machine learning?,3,ART_5c051946be0e_chunk_0096,ART_5c051946be0e,Learning,96,0.650521,https://en.wikipedia.org/wiki/Learning,0.017185
8,What is machine learning?,4,ART_e2f5507de1c9_chunk_0013,ART_e2f5507de1c9,Machine learning,13,0.648760,https://en.wikipedia.org/wiki/Machine_learning,0.017185
9,What is machine learning?,5,ART_483ea1c94a31_chunk_0000,ART_483ea1c94a31,Representation learning,0,0.645654,https://en.wikipedia.org/wiki/Representation_l...,0.017185


In [46]:
# ============================================================
# QUERY-LEVEL RETRIEVAL SUMMARY
# ============================================================

query_summary = (
    multi_query_df
    .groupby("query")
    .agg(
        top_similarity=("similarity_score", "max"),
        average_similarity=("similarity_score", "mean"),
        minimum_similarity=("similarity_score", "min"),
        unique_articles=("article_id", "nunique"),
        latency_seconds=("latency_seconds", "first")
    )
    .reset_index()
)

query_summary["top_similarity"] = (
    query_summary["top_similarity"].round(6)
)

query_summary["average_similarity"] = (
    query_summary["average_similarity"].round(6)
)

query_summary["minimum_similarity"] = (
    query_summary["minimum_similarity"].round(6)
)

query_summary["latency_seconds"] = (
    query_summary["latency_seconds"].round(6)
)

display(query_summary)

,query,top_similarity,average_similarity,minimum_similarity,unique_articles,latency_seconds
0,What is a large language model?,0.812848,0.723252,0.693775,5,0.017697
1,What is a vector database?,0.796315,0.618429,0.537504,2,0.018464
2,What is artificial intelligence?,0.780551,0.745425,0.714266,3,0.021215
3,What is computer vision?,0.787387,0.725280,0.689818,2,0.017285
4,What is deep learning?,0.822443,0.764699,0.721938,3,0.019383
5,What is machine learning?,0.693090,0.662166,0.645654,5,0.017185
6,What is natural language processing?,0.802069,0.729531,0.673988,3,0.021289
7,What is retrieval augmented generation?,0.724167,0.674910,0.643094,4,0.017814


In [47]:
# ============================================================
# STEP 12 VALIDATION
# ============================================================

assert len(query_summary) == len(test_queries)

assert len(multi_query_df) == (
    len(test_queries) * TOP_K
)

assert multi_query_df["chunk_id"].notna().all()
assert multi_query_df["article_id"].notna().all()
assert multi_query_df["title"].notna().all()
assert multi_query_df["source_url"].notna().all()

assert multi_query_df["similarity_score"].notna().all()
assert np.isfinite(
    multi_query_df["similarity_score"]
).all()

assert (
    multi_query_df["latency_seconds"] >= 0
).all()

# Every query must have exactly TOP_K results
results_per_query = (
    multi_query_df
    .groupby("query")
    .size()
)

assert (
    results_per_query == TOP_K
).all()

print("=" * 70)
print("STEP 12 – MULTI-QUERY RETRIEVAL VALIDATION")
print("=" * 70)

print("Queries tested          : PASSED")
print("Top-K consistency       : PASSED")
print("Retrieved records       : PASSED")
print("Chunk traceability      : PASSED")
print("Source preservation     : PASSED")
print("Similarity scores       : PASSED")
print("Latency measurements    : PASSED")

STEP 12 – MULTI-QUERY RETRIEVAL VALIDATION
Queries tested          : PASSED
Top-K consistency       : PASSED
Retrieved records       : PASSED
Chunk traceability      : PASSED
Source preservation     : PASSED
Similarity scores       : PASSED
Latency measurements    : PASSED


In [48]:
# ============================================================
# STEP 13 - RETRIEVAL ACCURACY EVALUATION
# STEP 13.1 – GROUND-TRUTH QUERIES
# ============================================================

evaluation_queries = [
    {
        "query": "What is artificial intelligence?",
        "expected_title": "Artificial intelligence"
    },
    {
        "query": "What is machine learning?",
        "expected_title": "Machine learning"
    },
    {
        "query": "What is deep learning?",
        "expected_title": "Deep learning"
    },
    {
        "query": "What is natural language processing?",
        "expected_title": "Natural language processing"
    },
    {
        "query": "What is a large language model?",
        "expected_title": "Large language model"
    },
    {
        "query": "What is a vector database?",
        "expected_title": "Vector database"
    },
    {
        "query": "What is retrieval augmented generation?",
        "expected_title": "Retrieval-augmented generation"
    },
    {
        "query": "What is computer vision?",
        "expected_title": "Computer vision"
    }
]

evaluation_df = pd.DataFrame(evaluation_queries)

print("=" * 80)
print("RETRIEVAL ACCURACY EVALUATION DATASET")
print("=" * 80)

print("Evaluation queries:", len(evaluation_df))

display(evaluation_df)

# ============================================================
# VERIFY EXPECTED ARTICLES
# ============================================================

available_titles = set(
    chunk_metadata_df["title"].dropna().unique()
)

evaluation_df["expected_title_exists"] = (
    evaluation_df["expected_title"]
    .isin(available_titles)
)

display(
    evaluation_df[
        [
            "query",
            "expected_title",
            "expected_title_exists"
        ]
    ]
)

# ============================================================
# EXPECTED ARTICLE VALIDATION
# ============================================================

assert evaluation_df["expected_title_exists"].all()

print(
    "Expected articles available: PASSED"
)

print(
    "Articles verified:",
    evaluation_df["expected_title_exists"].sum()
)

RETRIEVAL ACCURACY EVALUATION DATASET
Evaluation queries: 8


,query,expected_title
0,What is artificial intelligence?,Artificial intelligence
1,What is machine learning?,Machine learning
2,What is deep learning?,Deep learning
3,What is natural language processing?,Natural language processing
4,What is a large language model?,Large language model
5,What is a vector database?,Vector database
6,What is retrieval augmented generation?,Retrieval-augmented generation
7,What is computer vision?,Computer vision


,query,expected_title,expected_title_exists
0,What is artificial intelligence?,Artificial intelligence,True
1,What is machine learning?,Machine learning,True
2,What is deep learning?,Deep learning,True
3,What is natural language processing?,Natural language processing,True
4,What is a large language model?,Large language model,True
5,What is a vector database?,Vector database,True
6,What is retrieval augmented generation?,Retrieval-augmented generation,True
7,What is computer vision?,Computer vision,True


Expected articles available: PASSED
Articles verified: 8


In [49]:
# ============================================================
# RESOLVE EXPECTED ARTICLE IDs
# ============================================================

article_lookup = (
    chunk_metadata_df[
        ["article_id", "title"]
    ]
    .drop_duplicates()
)

evaluation_df = evaluation_df.merge(
    article_lookup,
    left_on="expected_title",
    right_on="title",
    how="left"
)

evaluation_df = evaluation_df.rename(
    columns={
        "article_id": "expected_article_id"
    }
)

evaluation_df = evaluation_df.drop(
    columns=["title"]
)

print("=" * 80)
print("GROUND-TRUTH ARTICLE MAPPING")
print("=" * 80)

display(evaluation_df)

assert evaluation_df["expected_article_id"].notna().all()
assert evaluation_df["expected_article_id"].nunique() == len(
    evaluation_df
)

print("Expected article IDs: PASSED")
print(
    "Unique expected articles:",
    evaluation_df["expected_article_id"].nunique()
)

GROUND-TRUTH ARTICLE MAPPING


,query,expected_title,expected_title_exists,expected_article_id
0,What is artificial intelligence?,Artificial intelligence,True,ART_ede7794b399f
1,What is machine learning?,Machine learning,True,ART_e2f5507de1c9
2,What is deep learning?,Deep learning,True,ART_c17cc45649e3
3,What is natural language processing?,Natural language processing,True,ART_11f009b214a5
4,What is a large language model?,Large language model,True,ART_492cad1f5ade
5,What is a vector database?,Vector database,True,ART_7b75734df6b2
6,What is retrieval augmented generation?,Retrieval-augmented generation,True,ART_124961cdef74
7,What is computer vision?,Computer vision,True,ART_4c38f733e5ee


Expected article IDs: PASSED
Unique expected articles: 8


In [50]:
# ============================================================
# STEP 13.4 – RETRIEVAL ACCURACY TEST
# ============================================================

accuracy_results = []

for _, evaluation_row in evaluation_df.iterrows():

    query_text = evaluation_row["query"]
    expected_article_id = evaluation_row["expected_article_id"]

    for top_k in [1, 3, 5]:

        results = basic_similarity_search(
            query=query_text,
            top_k=top_k
        )

        retrieved_article_ids = (
            results["article_id"]
            .tolist()
        )

        retrieved_chunk_ids = (
            results["chunk_id"]
            .tolist()
        )

        hit = (
            expected_article_id
            in retrieved_article_ids
        )

        accuracy_results.append({
            "query": query_text,
            "expected_article_id": expected_article_id,
            "top_k": top_k,
            "hit": hit,
            "retrieved_articles": len(
                set(retrieved_article_ids)
            ),
            "retrieved_chunks": len(
                set(retrieved_chunk_ids)
            )
        })

accuracy_results_df = pd.DataFrame(
    accuracy_results
)

print("=" * 80)
print("RETRIEVAL ACCURACY RESULTS")
print("=" * 80)

display(accuracy_results_df)

# ============================================================
# HIT@K CALCULATION
# ============================================================

hit_at_k = (
    accuracy_results_df
    .groupby("top_k")["hit"]
    .mean()
    .reset_index()
)

hit_at_k["hit_rate"] = (
    hit_at_k["hit"] * 100
)

hit_at_k = hit_at_k.drop(
    columns=["hit"]
)

print("=" * 80)
print("RETRIEVAL HIT@K")
print("=" * 80)

display(hit_at_k)

RETRIEVAL ACCURACY RESULTS


,query,expected_article_id,top_k,hit,retrieved_articles,retrieved_chunks
0,What is artificial intelligence?,ART_ede7794b399f,1,True,1,1
1,What is artificial intelligence?,ART_ede7794b399f,3,True,2,3
2,What is artificial intelligence?,ART_ede7794b399f,5,True,3,5
3,What is machine learning?,ART_e2f5507de1c9,1,False,1,1
4,What is machine learning?,ART_e2f5507de1c9,3,False,3,3
5,What is machine learning?,ART_e2f5507de1c9,5,True,5,5
6,What is deep learning?,ART_c17cc45649e3,1,False,1,1
7,What is deep learning?,ART_c17cc45649e3,3,True,3,3
8,What is deep learning?,ART_c17cc45649e3,5,True,3,5
9,What is natural language processing?,ART_11f009b214a5,1,True,1,1


RETRIEVAL HIT@K


,top_k,hit_rate
0,1,62.5
1,3,75.0
2,5,100.0


In [51]:
# ============================================================
# STEP 13 VALIDATION
# ============================================================

expected_rows = (
    len(evaluation_df) * 3
)

assert len(accuracy_results_df) == expected_rows

assert set(
    accuracy_results_df["top_k"].unique()
) == {1, 3, 5}

assert accuracy_results_df["hit"].notna().all()

assert accuracy_results_df["hit"].isin(
    [True, False]
).all()

assert len(hit_at_k) == 3

assert (
    hit_at_k["hit_rate"].between(0, 100)
).all()

print("=" * 70)
print("STEP 13 – RETRIEVAL ACCURACY VALIDATION")
print("=" * 70)

print("Evaluation queries       : PASSED")
print("Top-K evaluations        : PASSED")
print("Hit@1 calculation        : PASSED")
print("Hit@3 calculation        : PASSED")
print("Hit@5 calculation        : PASSED")
print("Accuracy range           : PASSED")

STEP 13 – RETRIEVAL ACCURACY VALIDATION
Evaluation queries       : PASSED
Top-K evaluations        : PASSED
Hit@1 calculation        : PASSED
Hit@3 calculation        : PASSED
Hit@5 calculation        : PASSED
Accuracy range           : PASSED


In [52]:
# ==========================================================
# STEP 14 -  FINAL RETRIEVAL EVALUATION REPORT
# ============================================================

phase7_report = {
    "total_articles": 128,
    "total_chunks": 4693,

    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimensions": 384,

    "vector_database": "ChromaDB",
    "collection": "wiki_chunks_minilm",

    "default_top_k": 5,

    "basic_similarity_search": "PASSED",

    "retrieved_context_validation": "PASSED",

    "multi_query_testing": {
        "queries": 8,
        "top_k": 5,
        "status": "PASSED"
    },

    "retrieval_latency": {
        "mean_seconds": round(overall_mean, 6),
        "median_seconds": round(overall_median, 6),
        "minimum_seconds": round(overall_min, 6),
        "maximum_seconds": round(overall_max, 6)
    },

    "retrieval_accuracy": {
        "evaluation_queries": len(evaluation_df),
        "hit_at_1_percent": float(
            hit_at_k.loc[
                hit_at_k["top_k"] == 1,
                "hit_rate"
            ].iloc[0]
        ),
        "hit_at_3_percent": float(
            hit_at_k.loc[
                hit_at_k["top_k"] == 3,
                "hit_rate"
            ].iloc[0]
        ),
        "hit_at_5_percent": float(
            hit_at_k.loc[
                hit_at_k["top_k"] == 5,
                "hit_rate"
            ].iloc[0]
        )
    },

    "validation_status": "PASSED"
}

print("=" * 80)
print("PHASE 7 – FINAL RETRIEVAL EVALUATION REPORT")
print("=" * 80)

import json

print(
    json.dumps(
        phase7_report,
        indent=4
    )
)

PHASE 7 – FINAL RETRIEVAL EVALUATION REPORT
{
    "total_articles": 128,
    "total_chunks": 4693,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimensions": 384,
    "vector_database": "ChromaDB",
    "collection": "wiki_chunks_minilm",
    "default_top_k": 5,
    "basic_similarity_search": "PASSED",
    "retrieved_context_validation": "PASSED",
    "multi_query_testing": {
        "queries": 8,
        "top_k": 5,
        "status": "PASSED"
    },
    "retrieval_latency": {
        "mean_seconds": 0.024369,
        "median_seconds": 0.023408,
        "minimum_seconds": 0.017771,
        "maximum_seconds": 0.044203
    },
    "retrieval_accuracy": {
        "evaluation_queries": 8,
        "hit_at_1_percent": 62.5,
        "hit_at_3_percent": 75.0,
        "hit_at_5_percent": 100.0
    },
    "validation_status": "PASSED"
}


In [53]:
# ============================================================
# SAVE  REPORT
# ============================================================

phase7_report_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retrieval"
    / "phase7_retrieval_report.json"
)

phase7_report_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    phase7_report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        phase7_report,
        f,
        indent=4
    )

print("Saved:")
print(phase7_report_path)

Saved:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\retrieval\phase7_retrieval_report.json
